In [ ]:
# -------------------------------
# Task 1: Plane heatmap generator
# -------------------------------

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Use GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------------
# Create a dummy 3D volume
# Intensities do not matter here because this task only tests the plane heatmap.
# -------------------------------
D, H, W = 64, 128, 128
volume = torch.zeros(D, H, W, device=device)

# Define the centre of the volume as the point on the plane
# Coordinates are stored in (z, y, x) order
p0 = torch.tensor([(D - 1) / 2, (H - 1) / 2, (W - 1) / 2], device=device)


def gaussian_kernel_3d(sigma: float, truncate: float = 3.0, device="cpu"):
    """
    Create a normalised 3D Gaussian kernel for blurring the plane mask.

    sigma controls how smooth the heatmap becomes.
    truncate controls the size of the kernel.
    """
    if sigma <= 0:
        raise ValueError("sigma must be > 0")

    # Work out the kernel radius and size
    radius = int(truncate * sigma + 0.5)
    size = 2 * radius + 1

    # Create the 1D Gaussian first
    coords = torch.arange(-radius, radius + 1, device=device).float()
    g1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g1d = g1d / g1d.sum()

    # Combine three 1D Gaussians to make a 3D kernel
    g3d = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]
    g3d = g3d / g3d.sum()

    # Reshape so it can be used by conv3d
    return g3d.view(1, 1, size, size, size)


def plane_heatmap_from_volume(
    volume: torch.Tensor,
    normal: torch.Tensor,
    point_on_plane: torch.Tensor | None = None,
    thickness: float = 1.0,
    sigma: float = 1.0,
    truncate: float = 3.0,
):
    """
    Create a 3D heatmap that highlights voxels close to a plane.

    The plane is defined using a normal vector and a point on the plane.
    A thin binary slab is first created, then blurred to make a smooth heatmap.
    """
    assert volume.ndim == 3, "volume must be (D,H,W)"
    device = volume.device
    D, H, W = volume.shape

    # Move the normal to the correct device and normalise it
    normal = normal.to(device).float()
    nrm = normal.norm()
    if nrm < 1e-8:
        raise ValueError("normal vector is near zero; cannot define a plane orientation.")
    normal = normal / nrm

    # If no point is given, use the centre of the volume
    if point_on_plane is None:
        point_on_plane = torch.tensor([(D - 1) / 2, (H - 1) / 2, (W - 1) / 2], device=device).float()
    else:
        point_on_plane = point_on_plane.to(device).float()

    # Create a voxel coordinate grid in (z, y, x) order
    z = torch.arange(D, device=device)
    y = torch.arange(H, device=device)
    x = torch.arange(W, device=device)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")
    coords = torch.stack([zz, yy, xx], dim=0).float()  # (3, D, H, W)

    # Calculate vector from the plane point to every voxel
    v = coords - point_on_plane.view(3, 1, 1, 1)

    # Project each voxel onto the normal direction to get distance from the plane
    signed_dist = (v * normal.view(3, 1, 1, 1)).sum(dim=0)  # (D,H,W)
    dist = signed_dist.abs()

    # Create a binary slab around the plane
    mask = (dist <= thickness).float()

    # Build the Gaussian kernel used to smooth the slab
    kernel = gaussian_kernel_3d(sigma=sigma, truncate=truncate, device=device)

    # Add batch and channel dimensions for conv3d
    mask_5d = mask.unsqueeze(0).unsqueeze(0)

    # Pad the mask so the output heatmap has the same size as the input
    kD, kH, kW = kernel.shape[-3:]
    pad = (kW // 2, kW // 2, kH // 2, kH // 2, kD // 2, kD // 2)

    # Replicate padding avoids dark edges after blurring
    mask_padded = F.pad(mask_5d, pad, mode="replicate")

    # Blur the binary slab to create the final soft heatmap
    heatmap = F.conv3d(mask_padded, kernel)[0, 0]  # back to (D,H,W)

    return heatmap, mask, dist


# -------------------------------
# Run test cases with different normals, sigma values and thicknesses
# -------------------------------
tests = [
    {"name": "Test 0: n=(1,0,0), sigma=1, thickness=1", "normal": (1, 0, 0), "sigma": 1, "thickness": 1},
    {"name": "Test 1: n=(0,1,0), sigma=1, thickness=1", "normal": (0, 1, 0), "sigma": 1, "thickness": 1},
    {"name": "Test 2: n=(1,0,0), sigma=3, thickness=1", "normal": (1, 0, 0), "sigma": 3, "thickness": 1},
    {"name": "Test 3: n=(1,0,0), sigma=1, thickness=3", "normal": (1, 0, 0), "sigma": 1, "thickness": 3},
]

results = []

# Generate one heatmap for each test case
for t in tests:
    n = torch.tensor(t["normal"], device=device).float()
    heatmap, mask, dist = plane_heatmap_from_volume(
        volume=volume,
        normal=n,
        point_on_plane=p0,
        thickness=float(t["thickness"]),
        sigma=float(t["sigma"]),
        truncate=3.0,
    )
    results.append((heatmap, mask, dist))

    # Print simple summary statistics for the heatmap
    print(
        t["name"],
        "| heatmap min/max/mean:",
        heatmap.min().item(),
        heatmap.max().item(),
        heatmap.mean().item()
    )


# -------------------------------
# Plot axial, coronal and sagittal views for each test
# -------------------------------
z_mid = D // 2
y_mid = H // 2
x_mid = W // 2

fig, axes = plt.subplots(nrows=len(tests), ncols=3, figsize=(14, 3.5 * len(tests)))

for i, (t, (heatmap, mask, dist)) in enumerate(zip(tests, results)):
    # Take central slices through the heatmap
    axial = heatmap[z_mid, :, :]      # (H,W)
    coronal = heatmap[:, y_mid, :]    # (D,W)
    sagittal = heatmap[:, :, x_mid]   # (D,H)

    # Plot axial view
    ax0 = axes[i, 0]
    im0 = ax0.imshow(axial.detach().cpu(), cmap="hot")
    ax0.set_title(f"{t['name']}\nAxial (z={z_mid})")
    ax0.axis("off")
    fig.colorbar(im0, ax=ax0, fraction=0.046, pad=0.04)

    # Plot coronal view
    ax1 = axes[i, 1]
    im1 = ax1.imshow(coronal.detach().cpu(), cmap="hot")
    ax1.set_title(f"Coronal (y={y_mid})")
    ax1.axis("off")
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # Plot sagittal view
    ax2 = axes[i, 2]
    im2 = ax2.imshow(sagittal.detach().cpu(), cmap="hot")
    ax2.set_title(f"Sagittal (x={x_mid})")
    ax2.axis("off")
    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# ============================================================
# Extra test: diagonal normal = (1,1,1)
# This checks that the function works for a non-axis-aligned plane.
# ============================================================

normal_diag = torch.tensor([1.0, 1.0, 1.0], device=device)  # (z,y,x)

# Generate heatmap for diagonal plane
heatmap_diag, mask_diag, dist_diag = plane_heatmap_from_volume(
    volume=volume,
    normal=normal_diag,
    point_on_plane=p0,
    thickness=1.0,
    sigma=1.0,
)

print("\nDiagonal normal test (1,1,1)")
print("Heatmap shape:", heatmap_diag.shape)
print("Min:", heatmap_diag.min().item())
print("Max:", heatmap_diag.max().item())

# ------------------------------------------------------------
# Numerical check:
# voxels with highest heatmap values should lie close to the plane
# ------------------------------------------------------------

# Flatten heatmap and distance volumes so they can be sorted easily
heat_flat = heatmap_diag.reshape(-1)
dist_flat = dist_diag.reshape(-1)

# Number of brightest voxels to inspect
topk = 20

# Find top-k brightest heatmap voxels
top_vals, top_idx = torch.topk(heat_flat, k=topk)

# Convert flat indices back to (z, y, x)
z_idx = top_idx // (H * W)
y_idx = (top_idx % (H * W)) // W
x_idx = top_idx % W

# Get the distance of these voxels from the target plane
top_dists = dist_flat[top_idx]

print(f"\nTop-{topk} heatmap voxels for diagonal normal test:")
print(" rank |   z |   y |   x | heatmap value | distance to plane")
for r in range(topk):
    print(
        f"{r+1:5d} | "
        f"{int(z_idx[r]):3d} | "
        f"{int(y_idx[r]):3d} | "
        f"{int(x_idx[r]):3d} | "
        f"{top_vals[r].item():13.6f} | "
        f"{top_dists[r].item():17.6f}"
    )

# Summarise how close the brightest voxels are to the target plane
print("\nDistance summary for top-k heatmap voxels:")
print("Min distance :", top_dists.min().item())
print("Max distance :", top_dists.max().item())
print("Mean distance:", top_dists.mean().item())

# ------------------------------------------------------------
# Visualise centred slices for the diagonal plane
# ------------------------------------------------------------

z_mid = D // 2
y_mid = H // 2
x_mid = W // 2

hm_ax = heatmap_diag[z_mid, :, :]
hm_co = heatmap_diag[:, y_mid, :]
hm_sa = heatmap_diag[:, :, x_mid]

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(hm_ax.cpu(), cmap="hot")
plt.title("Heatmap (1,1,1): Axial")
plt.colorbar()
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(hm_co.cpu(), cmap="hot", aspect="auto")
plt.title("Heatmap (1,1,1): Coronal")
plt.colorbar()
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(hm_sa.cpu(), cmap="hot", aspect="auto")
plt.title("Heatmap (1,1,1): Sagittal")
plt.colorbar()
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TASK 2: Cylinder + heatmap tests
# Part 1: test the heatmap on a cylinder volume
# Part 2: rotate the cylinder and check that the heatmap follows it
# Also includes numerical sanity checks and 90-degree rotations
# ============================================================

import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Use GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ------------------------------------------------------------
# Set volume size
# Shape is stored as (D,H,W) = (z,y,x)
# ------------------------------------------------------------
D, H, W = 64, 128, 128

# ------------------------------------------------------------
# Build a simple synthetic cylinder aligned with the z-axis
# ------------------------------------------------------------
cy = (H - 1) / 2.0
cx = (W - 1) / 2.0
cyl_radius = 30.0

# Create 2D coordinate grid in the y-x plane
y = torch.arange(H, device=device).float()
x = torch.arange(W, device=device).float()
yy, xx = torch.meshgrid(y, x, indexing="ij")

# Make a circular mask
r = torch.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
circle = (r <= cyl_radius).float()                 # (H,W)

# Repeat the circle through z to create a 3D cylinder
volume = circle.unsqueeze(0).repeat(D, 1, 1)       # (D,H,W)

# ------------------------------------------------------------
# Define the plane for the unrotated cylinder
# normal is in (z,y,x) order
# ------------------------------------------------------------
base_normal_zyx = torch.tensor([1.0, 0.0, 0.0], device=device)  # along z -> axial plane
base_p0 = torch.tensor([(D - 1) / 2.0, (H - 1) / 2.0, (W - 1) / 2.0], device=device)

# ============================================================
# HEATMAP FUNCTION
# Same heatmap generation method from Task 1
# ============================================================

def gaussian_kernel_3d(sigma: float, truncate: float = 3.0, device="cpu"):
    # Create a 3D Gaussian kernel for smoothing the binary plane mask
    if sigma <= 0:
        raise ValueError("sigma must be > 0")

    # Work out the kernel size from sigma
    radius = int(truncate * sigma + 0.5)
    size = 2 * radius + 1

    # Create a 1D Gaussian
    coords = torch.arange(-radius, radius + 1, device=device).float()
    g1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g1d = g1d / g1d.sum()

    # Combine three 1D Gaussians to create a 3D Gaussian
    g3d = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]
    g3d = g3d / g3d.sum()

    # Reshape for conv3d
    return g3d.view(1, 1, size, size, size)


def plane_heatmap_from_volume(
    volume: torch.Tensor,
    normal: torch.Tensor,
    point_on_plane: torch.Tensor | None = None,
    thickness: float = 1.0,
    sigma: float = 1.0,
    truncate: float = 3.0,
):
    """
    Create a 3D heatmap highlighting voxels close to a plane.

    volume: (D,H,W) in (z,y,x)
    normal: (3,) in (z,y,x)
    """
    assert volume.ndim == 3, "volume must be (D,H,W)"
    device = volume.device
    D, H, W = volume.shape

    # Normalise the plane normal
    normal = normal.to(device).float()
    nrm = normal.norm()
    if nrm < 1e-8:
        raise ValueError("normal vector near zero")
    normal = normal / nrm

    # Use volume centre if no plane point is given
    if point_on_plane is None:
        point_on_plane = torch.tensor(
            [(D - 1) / 2, (H - 1) / 2, (W - 1) / 2],
            device=device
        ).float()
    else:
        point_on_plane = point_on_plane.to(device).float()

    # Create coordinate grid in (z,y,x)
    z = torch.arange(D, device=device)
    y = torch.arange(H, device=device)
    x = torch.arange(W, device=device)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")

    # Stack coordinates into one tensor
    coords = torch.stack([zz, yy, xx], dim=0).float()  # (3,D,H,W)

    # Vector from the plane point to every voxel
    v = coords - point_on_plane.view(3, 1, 1, 1)

    # Calculate distance of each voxel from the plane
    signed_dist = (v * normal.view(3, 1, 1, 1)).sum(dim=0)
    dist = signed_dist.abs()

    # Create binary slab around the plane
    mask = (dist <= thickness).float()

    # Smooth the slab into a heatmap
    kernel = gaussian_kernel_3d(sigma=sigma, truncate=truncate, device=device)
    mask_5d = mask.unsqueeze(0).unsqueeze(0)  # (1,1,D,H,W)

    # Pad so that the output stays the same size
    kD, kH, kW = kernel.shape[-3:]
    pad = (kW // 2, kW // 2, kH // 2, kH // 2, kD // 2, kD // 2)
    mask_padded = F.pad(mask_5d, pad, mode="replicate")

    # Apply Gaussian blur
    heatmap = F.conv3d(mask_padded, kernel)[0, 0]  # (D,H,W)

    return heatmap, mask, dist


# ============================================================
# TASK 2 PART 1
# Test the heatmap function on the unrotated cylinder
# ============================================================

heatmap, mask, dist = plane_heatmap_from_volume(
    volume=volume,
    normal=base_normal_zyx,
    point_on_plane=base_p0,
    thickness=1.0,
    sigma=1.0,
)

# Print basic shape and intensity information
print("Part 1")
print("heatmap:", heatmap.shape, heatmap.min().item(), heatmap.max().item())
print("volume:", volume.shape, volume.min().item(), volume.max().item())

# ------------------------------------------------------------
# Visualise central slices
# ------------------------------------------------------------
z_mid = int(round((D - 1) / 2.0))
y_mid = int(round(cy))
x_mid = int(round(cx))

# Extract central slices from the cylinder
vol_ax = volume[z_mid, :, :]
vol_co = volume[:, y_mid, :]
vol_sa = volume[:, :, x_mid]

# Extract matching slices from the heatmap
hm_ax = heatmap[z_mid, :, :]
hm_co = heatmap[:, y_mid, :]
hm_sa = heatmap[:, :, x_mid]

plt.figure(figsize=(18, 10))

# Row 1: cylinder views
plt.subplot(3, 3, 1)
plt.imshow(vol_ax.cpu(), cmap="gray", vmin=0, vmax=1)
plt.title("Cylinder: Axial (z)")
plt.axis("off")

plt.subplot(3, 3, 2)
plt.imshow(vol_co.cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
plt.title("Cylinder: Coronal (y)")
plt.axis("off")

plt.subplot(3, 3, 3)
plt.imshow(vol_sa.cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
plt.title("Cylinder: Sagittal (x)")
plt.axis("off")

# Row 2: heatmap views
plt.subplot(3, 3, 4)
plt.imshow(hm_ax.cpu(), cmap="hot")
plt.title("Heatmap: Axial (z)")
plt.colorbar()
plt.axis("off")

plt.subplot(3, 3, 5)
plt.imshow(hm_co.cpu(), cmap="hot", aspect="auto")
plt.title("Heatmap: Coronal (y)")
plt.colorbar()
plt.axis("off")

plt.subplot(3, 3, 6)
plt.imshow(hm_sa.cpu(), cmap="hot", aspect="auto")
plt.title("Heatmap: Sagittal (x)")
plt.colorbar()
plt.axis("off")

# Row 3: cylinder and heatmap overlays
plt.subplot(3, 3, 7)
plt.imshow(vol_ax.cpu(), cmap="gray", vmin=0, vmax=1)
plt.imshow(hm_ax.cpu(), cmap="hot", alpha=0.6)
plt.title("Overlay (Axial): volume + heatmap")
plt.axis("off")

plt.subplot(3, 3, 8)
plt.imshow(vol_co.cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
plt.imshow(hm_co.cpu(), cmap="hot", alpha=0.6, aspect="auto")
plt.title("Overlay (Coronal): volume + heatmap")
plt.axis("off")

plt.subplot(3, 3, 9)
plt.imshow(vol_sa.cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
plt.imshow(hm_sa.cpu(), cmap="hot", alpha=0.6, aspect="auto")
plt.title("Overlay (Sagittal): volume + heatmap")
plt.axis("off")

plt.tight_layout()
plt.show()


# ============================================================
# TASK 2 PART 2
# Rotate the cylinder and check that the heatmap rotates correctly
# ============================================================

# ------------------------------------------------------------
# Rotation utilities
# Important: PyTorch grid_sample uses (x,y,z), but the volume is stored as (z,y,x)
# ------------------------------------------------------------
def rot_matrix_xyz(rx_deg=0.0, ry_deg=0.0, rz_deg=0.0, device="cpu"):
    """Rotation matrix in (x,y,z) order."""

    # Convert degrees to radians
    rx = math.radians(rx_deg)
    ry = math.radians(ry_deg)
    rz = math.radians(rz_deg)

    # Precompute sine and cosine values
    cx, sx = math.cos(rx), math.sin(rx)
    cy, sy = math.cos(ry), math.sin(ry)
    cz, sz = math.cos(rz), math.sin(rz)

    # Rotation about x-axis
    Rx = torch.tensor([
        [1,  0,   0],
        [0, cx, -sx],
        [0, sx,  cx],
    ], device=device, dtype=torch.float32)

    # Rotation about y-axis
    Ry = torch.tensor([
        [ cy, 0, sy],
        [  0, 1,  0],
        [-sy, 0, cy],
    ], device=device, dtype=torch.float32)

    # Rotation about z-axis
    Rz = torch.tensor([
        [cz, -sz, 0],
        [sz,  cz, 0],
        [ 0,   0, 1],
    ], device=device, dtype=torch.float32)

    # Combined rotation matrix
    return (Rz @ Ry @ Rx)


def rotate_volume_3d_xyz(volume_zyx: torch.Tensor, R_xyz: torch.Tensor, mode="bilinear"):
    """
    Rotate a volume stored as (D,H,W)=(z,y,x) using affine_grid/grid_sample.
    R_xyz is 3x3 in (x,y,z).
    """
    assert volume_zyx.ndim == 3
    device = volume_zyx.device

    # Add batch and channel dimensions
    vol = volume_zyx.unsqueeze(0).unsqueeze(0).float()  # (1,1,D,H,W)

    # Create affine matrix
    A = torch.zeros((1, 3, 4), device=device, dtype=torch.float32)
    A[0, :, :3] = R_xyz.to(device)

    # Create sampling grid and sample the rotated volume
    grid = F.affine_grid(A, size=vol.shape, align_corners=False)
    rot = F.grid_sample(vol, grid, mode=mode, padding_mode="zeros", align_corners=False)

    # Remove batch and channel dimensions
    return rot[0, 0]  # (D,H,W)


def zyx_to_xyz(v_zyx: torch.Tensor):
    """Convert vector order from (z,y,x) to (x,y,z)."""
    return torch.stack([v_zyx[2], v_zyx[1], v_zyx[0]])


def xyz_to_zyx(v_xyz: torch.Tensor):
    """Convert vector order from (x,y,z) to (z,y,x)."""
    return torch.stack([v_xyz[2], v_xyz[1], v_xyz[0]])


# ------------------------------------------------------------
# Helper function for plotting rotated volume and heatmap
# ------------------------------------------------------------
def plot_volume_and_heatmap(volume_rot: torch.Tensor, heatmap: torch.Tensor, title_prefix: str):

    # Get volume shape
    D, H, W = volume_rot.shape

    # Central slice indices
    z_mid = int(round((D - 1) / 2.0))
    y_mid = int(round((H - 1) / 2.0))
    x_mid = int(round((W - 1) / 2.0))

    # Extract central slices from the rotated volume
    vol_ax = volume_rot[z_mid]
    vol_co = volume_rot[:, y_mid, :]
    vol_sa = volume_rot[:, :, x_mid]

    # Extract matching central slices from the heatmap
    hm_ax = heatmap[z_mid]
    hm_co = heatmap[:, y_mid, :]
    hm_sa = heatmap[:, :, x_mid]

    plt.figure(figsize=(18, 7))

    # Row 1: rotated volume
    plt.subplot(2, 3, 1)
    plt.imshow(vol_ax.detach().cpu(), cmap="gray", vmin=0, vmax=1)
    plt.title(f"{title_prefix} Volume: Axial (z)")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(vol_co.detach().cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
    plt.title(f"{title_prefix} Volume: Coronal (y)")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(vol_sa.detach().cpu(), cmap="gray", vmin=0, vmax=1, aspect="auto")
    plt.title(f"{title_prefix} Volume: Sagittal (x)")
    plt.axis("off")

    # Row 2: heatmap for the rotated plane
    plt.subplot(2, 3, 4)
    plt.imshow(hm_ax.detach().cpu(), cmap="hot")
    plt.title(f"{title_prefix} Heatmap: Axial (z)")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(2, 3, 5)
    plt.imshow(hm_co.detach().cpu(), cmap="hot", aspect="auto")
    plt.title(f"{title_prefix} Heatmap: Coronal (y)")
    plt.colorbar()
    plt.axis("off")

    plt.subplot(2, 3, 6)
    plt.imshow(hm_sa.detach().cpu(), cmap="hot", aspect="auto")
    plt.title(f"{title_prefix} Heatmap: Sagittal (x)")
    plt.colorbar()
    plt.axis("off")

    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Rotation tests
# These include smaller rotations and 90-degree sanity checks
# ------------------------------------------------------------
tests = [
    {"name": "Test 1: Rotate about y by +30°", "rx": 0,  "ry": 30, "rz": 0},
    {"name": "Test 2: Rotate about x by +25°", "rx": 25, "ry": 0,  "rz": 0},
    {"name": "Test 3: Rotate about z by +45°", "rx": 0,  "ry": 0,  "rz": 45},

    # 90-degree tests are useful because the expected output is easier to check visually
    {"name": "Test 4: Rotate about y by +90°", "rx": 0,  "ry": 90, "rz": 0},
    {"name": "Test 5: Rotate about x by +90°", "rx": 90, "ry": 0,  "rz": 0},
    {"name": "Test 6: Rotate about z by +90°", "rx": 0,  "ry": 0,  "rz": 90},
]

# Heatmap settings
THICKNESS = 1.0
SIGMA = 1.0

# Number of brightest voxels used for the sanity check
TOPK = 500

print("\nPart 2")

# ------------------------------------------------------------
# Run each rotation test
# ------------------------------------------------------------
for t in tests:

    # Build rotation matrix for this test
    R_xyz = rot_matrix_xyz(t["rx"], t["ry"], t["rz"], device=device)

    # Rotate the cylinder volume
    volume_rot = rotate_volume_3d_xyz(volume, R_xyz, mode="bilinear")

    # Rotate the plane normal using the same rotation
    normal_xyz = zyx_to_xyz(base_normal_zyx).to(device)
    normal_rot_xyz = R_xyz @ normal_xyz
    normal_rot_zyx = xyz_to_zyx(normal_rot_xyz)

    # Generate a heatmap using the rotated normal
    heatmap_rot, mask_rot, dist_rot = plane_heatmap_from_volume(
        volume=volume_rot,
        normal=normal_rot_zyx,
        point_on_plane=base_p0,
        thickness=THICKNESS,
        sigma=SIGMA,
        truncate=3.0,
    )

    # Print the rotated normal for checking
    print(f"{t['name']} | normal_rot_zyx = {normal_rot_zyx.tolist()}")

    # Numerical sanity check:
    # brightest heatmap voxels should be close to the target plane
    with torch.no_grad():
        heat_flat = heatmap_rot.flatten()
        dist_flat = dist_rot.flatten()

        topk_vals, topk_idx = torch.topk(heat_flat, TOPK)
        topk_dist = dist_flat[topk_idx]

        print(f"  Mean distance of top-{TOPK} brightest voxels: {topk_dist.mean().item():.6f}")
        print(f"  Max  distance of top-{TOPK} brightest voxels: {topk_dist.max().item():.6f}")
        print("")

    # Plot the rotated volume and corresponding heatmap
    plot_volume_and_heatmap(volume_rot, heatmap_rot, title_prefix=t["name"])

In [ ]:
# ============================================================
# TASK 3 (AMENDED): Train a 3D UNet to map 3D volume -> 3D heatmap
# - Training samples: rotated cylinder volumes
# - Ground truth heatmap: generated using Task-1 function (plane mask -> Gaussian blur)
# - Cross-validation for hyperparams
# - Train/Val loss curves
# - Explicit TRAIN / VAL / TEST L1 + MSE metrics
# - Visualise 20 training cases + 20 test cases
# - Show angle error for BOTH training and test cases
# - Confirm GT correctness via sanity plots + normal agreement check
# ============================================================

import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


# -----------------------------
# Device + reproducibility
# -----------------------------
# Use GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seeds so results are more reproducible
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


# ============================================================
# 1) TASK-1 GROUND TRUTH: plane_heatmap_from_volume
#    (distance -> mask -> Gaussian blur)
#    NOTE: normal is in (z,y,x) order for THIS function.
# ============================================================

def gaussian_kernel_3d(sigma: float, truncate: float = 3.0, device="cpu"):
    # Work out the kernel size from sigma
    radius = int(truncate * sigma + 0.5)
    size = 2 * radius + 1

    # Create 1D Gaussian values
    coords = torch.arange(-radius, radius + 1, device=device).float()
    g1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g1d = g1d / (g1d.sum() + 1e-12)

    # Convert the 1D Gaussian into a 3D Gaussian kernel
    g3d = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]
    g3d = g3d / (g3d.sum() + 1e-12)

    # Return in the shape expected by conv3d
    return g3d.view(1, 1, size, size, size)

def plane_heatmap_from_volume(
    volume: torch.Tensor,
    normal_zyx: torch.Tensor,
    point_on_plane_zyx: torch.Tensor | None = None,
    thickness: float = 1.0,
    sigma: float = 1.0,
    truncate: float = 3.0,
    align_corners: bool = False,  # not used here, but kept for consistency
):
    """
    volume: (D,H,W)
    normal_zyx: (3,) in (z,y,x)
    point_on_plane_zyx: (3,) in (z,y,x)
    returns heatmap (D,H,W) in [0, ~1]
    """
    assert volume.ndim == 3, "volume must be (D,H,W)"
    dev = volume.device
    D, H, W = volume.shape

    # Normalise the plane normal
    n = normal_zyx.to(dev).float()
    n = n / (n.norm() + 1e-8)

    # If no point is given, use the centre of the volume
    if point_on_plane_zyx is None:
        p0 = torch.tensor([(D - 1) / 2, (H - 1) / 2, (W - 1) / 2], device=dev).float()
    else:
        p0 = point_on_plane_zyx.to(dev).float()

    # Create voxel coordinate grids
    z = torch.arange(D, device=dev).float()
    y = torch.arange(H, device=dev).float()
    x = torch.arange(W, device=dev).float()
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")

    # Stack coordinates in (z,y,x) order
    coords = torch.stack([zz, yy, xx], dim=0)  # (3,D,H,W)

    # Vector from the plane point to every voxel
    v = coords - p0.view(3, 1, 1, 1)

    # Calculate distance of each voxel from the plane
    signed_dist = (v * n.view(3, 1, 1, 1)).sum(dim=0)
    dist = signed_dist.abs()

    # Create a binary slab around the plane
    mask = (dist <= thickness).float()  # binary plane slab

    # Blur the binary mask to make a smooth heatmap
    kernel = gaussian_kernel_3d(sigma=sigma, truncate=truncate, device=dev)
    mask_5d = mask[None, None]  # (1,1,D,H,W)

    # Pad before convolution so output size stays the same
    kD, kH, kW = kernel.shape[-3:]
    pad = (kW//2, kW//2, kH//2, kH//2, kD//2, kD//2)
    mask_padded = F.pad(mask_5d, pad, mode="replicate")

    # Apply Gaussian smoothing
    heat = F.conv3d(mask_padded, kernel)[0, 0]  # (D,H,W)

    # Normalise to [0,1] for stable learning
    heat = heat / (heat.max().clamp_min(1e-8))
    return heat


# ============================================================
# 2) Synthetic volume: cylinder + gradient + bump (asymmetry)
# ============================================================

def make_cylinder_with_bump_and_gradient(D, H, W, cyl_radius=18.0, bump_radius=5.0, device="cpu"):
    # Centre of the cylinder in the y-x plane
    cy = (H - 1) / 2
    cx = (W - 1) / 2

    # Create 2D y-x grid
    y = torch.arange(H, device=device).float()
    x = torch.arange(W, device=device).float()
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    # Create a circular cross-section
    r = torch.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    circle = (r <= cyl_radius).float()

    # Repeat circle along z to make a cylinder
    cyl = circle[None].repeat(D, 1, 1)  # (D,H,W)

    # Add gradient in x direction to break symmetry
    grad2d = (xx / (W - 1)).clamp(0, 1)
    cyl = cyl * (0.35 + 0.65 * grad2d[None].repeat(D, 1, 1))

    # Position of the offset bump
    bz = (D - 1) * 0.65
    by = cy - 0.35 * cyl_radius
    bx = cx + 0.35 * cyl_radius

    # Create a small 3D bump
    zz = torch.arange(D, device=device).float()[:, None, None]
    bump_dist = torch.sqrt((zz - bz) ** 2 + (yy[None] - by) ** 2 + (xx[None] - bx) ** 2)
    bump = (bump_dist <= bump_radius).float()

    # Combine cylinder and bump, keeping intensities between 0 and 1
    return torch.clamp(cyl + bump, 0.0, 1.0)


# ============================================================
# 3) Rotation utilities (PyTorch affine_grid/grid_sample uses (x,y,z))
#    Our volume tensor is stored as (D,H,W) == (z,y,x).
# ============================================================

def rot_matrix_xyz(rx_deg=0.0, ry_deg=0.0, rz_deg=0.0, device="cpu"):
    # Convert degrees to radians
    rx = math.radians(rx_deg)
    ry = math.radians(ry_deg)
    rz = math.radians(rz_deg)

    # Pre-compute sin and cos values
    cx, sx = math.cos(rx), math.sin(rx)
    cy, sy = math.cos(ry), math.sin(ry)
    cz, sz = math.cos(rz), math.sin(rz)

    # Rotation around x-axis
    Rx = torch.tensor([
        [1,  0,   0],
        [0, cx, -sx],
        [0, sx,  cx],
    ], device=device, dtype=torch.float32)

    # Rotation around y-axis
    Ry = torch.tensor([
        [ cy, 0, sy],
        [  0, 1,  0],
        [-sy, 0, cy],
    ], device=device, dtype=torch.float32)

    # Rotation around z-axis
    Rz = torch.tensor([
        [cz, -sz, 0],
        [sz,  cz, 0],
        [ 0,   0, 1],
    ], device=device, dtype=torch.float32)

    # Combined rotation matrix
    return (Rz @ Ry @ Rx)

def rotate_volume_3d_xyz(volume_zyx: torch.Tensor, R_xyz: torch.Tensor, align_corners: bool = False):
    """
    volume_zyx: (D,H,W) == (z,y,x)
    R_xyz: rotation matrix for (x,y,z)
    """
    # Add batch and channel dimensions
    vol = volume_zyx[None, None].float()  # (1,1,D,H,W)

    # Build affine transform matrix
    A = torch.zeros((1, 3, 4), device=volume_zyx.device, dtype=torch.float32)
    A[0, :, :3] = R_xyz.to(volume_zyx.device)

    # Create sampling grid and rotate the volume
    grid = F.affine_grid(A, size=vol.shape, align_corners=align_corners)
    rot = F.grid_sample(vol, grid, mode="bilinear", padding_mode="zeros", align_corners=align_corners)

    return rot[0, 0]

def xyz_to_zyx(v_xyz: torch.Tensor):
    # Convert vector order from (x,y,z) to (z,y,x)
    return torch.stack([v_xyz[2], v_xyz[1], v_xyz[0]], dim=0)


# ============================================================
# 4) Dataset: fixed rotations -> (volume, heatmap_gt)
#    IMPORTANT: UNet trains on heatmap targets (NOT masks).
# ============================================================

def make_angles_deg(n, angle_range_degrees, seed_offset=0):
    # Generate a fixed list of random rotation angles
    rng = random.Random(seed + seed_offset)
    a = angle_range_degrees
    return [(rng.uniform(-a, a), rng.uniform(-a, a), rng.uniform(-a, a)) for _ in range(n)]

class RotatedHeatmapDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        angles_deg_list,
        D, H, W,
        thickness=1.5,
        sigma=1.0,
        truncate=3.0,
        align_corners=False,
        device="cpu"
    ):
        # Store dataset settings
        self.angles = angles_deg_list
        self.D, self.H, self.W = D, H, W
        self.thickness = thickness
        self.sigma = sigma
        self.truncate = truncate
        self.align_corners = align_corners
        self.device = device

        # Create one base volume, then rotate it for each sample
        self.base_vol = make_cylinder_with_bump_and_gradient(D, H, W, device=device)

        # Fixed centre point for the plane
        self.p0_zyx = torch.tensor([(D-1)/2, (H-1)/2, (W-1)/2], device=device).float()

        # Base normal = along z in (z,y,x) => [1,0,0]
        # In (x,y,z) that's [0,0,1]
        self.base_normal_xyz = torch.tensor([0.0, 0.0, 1.0], device=device).float()

    def __len__(self):
        # Number of rotated samples
        return len(self.angles)

    def __getitem__(self, idx):
        # Get rotation angles for this sample
        rx, ry, rz = self.angles[idx]
        R = rot_matrix_xyz(rx, ry, rz, device=self.device)

        # Rotate the synthetic volume
        vol_rot = rotate_volume_3d_xyz(self.base_vol, R, align_corners=self.align_corners)

        # Rotate normal consistently in xyz then convert to zyx
        n_rot_xyz = (R @ self.base_normal_xyz)
        n_rot_zyx = xyz_to_zyx(n_rot_xyz)

        # Generate the GT heatmap for the rotated plane
        heat_gt = plane_heatmap_from_volume(
            volume=vol_rot,
            normal_zyx=n_rot_zyx,
            point_on_plane_zyx=self.p0_zyx,
            thickness=self.thickness,
            sigma=self.sigma,
            truncate=self.truncate,
        )

        # Return with channel dim (C=1)
        return vol_rot.unsqueeze(0), heat_gt.unsqueeze(0), n_rot_zyx


# ============================================================
# 5) 3D UNet (bigger for accuracy)
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        # Two 3D convolution blocks with normalisation and activation
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base=24):
        super().__init__()

        # Encoder path
        self.down1 = DoubleConv(in_channels, base)
        self.pool1 = nn.MaxPool3d(2)

        self.down2 = DoubleConv(base, base*2)
        self.pool2 = nn.MaxPool3d(2)

        self.down3 = DoubleConv(base*2, base*4)
        self.pool3 = nn.MaxPool3d(2)

        self.down4 = DoubleConv(base*4, base*8)
        self.pool4 = nn.MaxPool3d(2)

        # Bottleneck layer
        self.bottleneck = DoubleConv(base*8, base*16)

        # Decoder path
        self.up4 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv4 = DoubleConv(base*16 + base*8, base*8)

        self.up3 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv3 = DoubleConv(base*8 + base*4, base*4)

        self.up2 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv2 = DoubleConv(base*4 + base*2, base*2)

        self.up1 = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)
        self.conv1 = DoubleConv(base*2 + base, base)

        # Final 1x1 convolution to output one heatmap channel
        self.out = nn.Conv3d(base, out_channels, 1)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x); p1 = self.pool1(d1)
        d2 = self.down2(p1); p2 = self.pool2(d2)
        d3 = self.down3(p2); p3 = self.pool3(d3)
        d4 = self.down4(p3); p4 = self.pool4(d4)

        # Bottleneck
        b = self.bottleneck(p4)

        # Decoder with skip connections
        u4 = self.up4(b); u4 = torch.cat([u4, d4], dim=1); u4 = self.conv4(u4)
        u3 = self.up3(u4); u3 = torch.cat([u3, d3], dim=1); u3 = self.conv3(u3)
        u2 = self.up2(u3); u2 = torch.cat([u2, d2], dim=1); u2 = self.conv2(u2)
        u1 = self.up1(u2); u1 = torch.cat([u1, d1], dim=1); u1 = self.conv1(u1)

        return self.out(u1)  # logits


# ============================================================
# 6) Loss (heatmap regression): L1 + MSE + gradient loss
# ============================================================

def gradient_3d(x):
    # x: (B,1,D,H,W)
    # Calculate simple finite differences in z, y and x
    dz = x[:, :, 1:] - x[:, :, :-1]
    dy = x[:, :, :, 1:] - x[:, :, :, :-1]
    dx = x[:, :, :, :, 1:] - x[:, :, :, :, :-1]
    return dz, dy, dx

def heatmap_loss(pred_logits, gt_heat, w_l1=0.7, w_mse=0.3, w_grad=0.1):
    # Convert logits to predicted heatmap values between 0 and 1
    pred = torch.sigmoid(pred_logits)

    # Main voxel-wise losses
    l1 = F.l1_loss(pred, gt_heat)
    mse = F.mse_loss(pred, gt_heat)

    # Gradient loss encourages similar heatmap shape and edges
    dz_p, dy_p, dx_p = gradient_3d(pred)
    dz_g, dy_g, dx_g = gradient_3d(gt_heat)

    grad = (F.l1_loss(dz_p, dz_g) + F.l1_loss(dy_p, dy_g) + F.l1_loss(dx_p, dx_g)) / 3.0

    # Weighted total loss
    return w_l1*l1 + w_mse*mse + w_grad*grad


# ============================================================
# 7) Task 4 helper for sanity: estimate normal from heatmap
# ============================================================

def estimate_plane_from_heatmap(heatmap_zyx: torch.Tensor, topk=20000, eps=1e-8):
    # heatmap_zyx: (D,H,W)
    assert heatmap_zyx.ndim == 3
    dev = heatmap_zyx.device
    D, H, W = heatmap_zyx.shape

    # Flatten heatmap into weights
    w = heatmap_zyx.reshape(-1)

    # Use only top-k brightest voxels if requested
    if topk is not None and topk < w.numel():
        vals, idx = torch.topk(w, k=topk)
        w = vals.clamp_min(0.0)

        # Convert flat indices back to z,y,x coordinates
        z = (idx // (H*W)).float()
        y = ((idx % (H*W)) // W).float()
        x = (idx % W).float()
        coords = torch.stack([z, y, x], dim=1)
    else:
        # Otherwise use all voxels
        z = torch.arange(D, device=dev)
        y = torch.arange(H, device=dev)
        x = torch.arange(W, device=dev)
        zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")
        coords = torch.stack([zz, yy, xx], dim=-1).reshape(-1, 3).float()
        w = w.clamp_min(0.0)

    # Normalise heatmap values into weights
    w = w / (w.sum() + eps)

    # Weighted centre point of the bright voxels
    point = (w[:, None] * coords).sum(dim=0)

    # Weighted covariance matrix
    X = coords - point[None, :]
    C = (w[:, None, None] * (X[:, :, None] * X[:, None, :])).sum(dim=0)

    # Smallest eigenvector gives the plane normal
    evals, evecs = torch.linalg.eigh(C)
    normal = evecs[:, 0]
    normal = normal / (normal.norm() + eps)

    return normal, point

def angle_between(u, v, eps=1e-8):
    # Normalise both vectors
    u = u / (u.norm() + eps)
    v = v / (v.norm() + eps)

    # Use absolute dot product because normal direction sign does not matter
    c = torch.clamp(torch.abs((u*v).sum()), 0.0, 1.0)  # sign-invariant

    # Convert angle from radians to degrees
    return float(torch.rad2deg(torch.acos(c)))


# ============================================================
# 8) Training loop + logging train/val curves
# ============================================================

def train_one_run(
    train_loader,
    val_loader,
    *,
    base=24,
    lr=2e-4,
    max_epochs=80,
    patience=12,
    w_l1=0.7,
    w_mse=0.3,
    w_grad=0.1
):
    # Create model and optimiser
    model = UNet3D(base=base).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # Reduce learning rate if validation loss stops improving
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=4
    )

    # Mixed precision scaler for CUDA
    scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))

    # Store losses for plotting
    train_curve = []
    val_curve = []

    # Track best validation model
    best_val = float("inf")
    best_state = None
    no_improve = 0

    prev_lr = opt.param_groups[0]["lr"]

    for epoch in range(1, max_epochs + 1):
        # ---------------- Training ----------------
        model.train()
        tr = 0.0

        for X, heat_gt, _n in train_loader:
            X = X.to(device)
            heat_gt = heat_gt.to(device)

            opt.zero_grad(set_to_none=True)

            # Forward pass with mixed precision if using GPU
            with torch.amp.autocast(device_type="cuda", enabled=(device=="cuda")):
                logits = model(X)
                loss = heatmap_loss(logits, heat_gt, w_l1=w_l1, w_mse=w_mse, w_grad=w_grad)

            # Backpropagation
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr += float(loss.item())

        # Average training loss for this epoch
        tr /= len(train_loader)
        train_curve.append(tr)

        # ---------------- Validation ----------------
        model.eval()
        va = 0.0
        with torch.no_grad():
            for X, heat_gt, _n in val_loader:
                X = X.to(device)
                heat_gt = heat_gt.to(device)

                with torch.amp.autocast(device_type="cuda", enabled=(device=="cuda")):
                    logits = model(X)
                    loss = heatmap_loss(logits, heat_gt, w_l1=w_l1, w_mse=w_mse, w_grad=w_grad)

                va += float(loss.item())

        # Average validation loss for this epoch
        va /= len(val_loader)
        val_curve.append(va)

        # Update scheduler using validation loss
        sched.step(va)

        # Report learning rate changes
        cur_lr = opt.param_groups[0]["lr"]
        if cur_lr != prev_lr:
            print(f"  LR reduced: {prev_lr:.2e} -> {cur_lr:.2e}")
            prev_lr = cur_lr

        print(f"Epoch {epoch:03d} | TrainLoss {tr:.6f} | ValLoss {va:.6f} | LR {cur_lr:.2e}")

        # Save best model state
        if va < best_val - 1e-5:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

            # Stop early if validation loss has not improved
            if no_improve >= patience:
                print("Early stopping.")
                break

    # Load best model before returning
    model.load_state_dict(best_state)
    model.to(device)

    return model, best_val, train_curve, val_curve


# ============================================================
# 9) Cross-validation (simple grid)
# ============================================================

def run_cv(ds_pool, k=3, grid=None, batch_size=2):
    # Default hyperparameter grid
    if grid is None:
        grid = [
            {"base": 24, "lr": 2e-4},
            {"base": 24, "lr": 1e-4},
            {"base": 32, "lr": 2e-4},
        ]

    # Shuffle dataset indices
    idx = list(range(len(ds_pool)))
    random.Random(seed).shuffle(idx)

    # Split into folds
    folds = np.array_split(idx, k)

    results = []

    # Try each hyperparameter setting
    for cfg in grid:
        print("\n" + "="*70)
        print("CV config:", cfg)
        print("="*70)
        fold_losses = []

        # Run k-fold cross-validation
        for fi in range(k):
            val_idx = folds[fi].tolist()
            train_idx = np.concatenate([folds[j] for j in range(k) if j != fi]).tolist()

            # Training fold loader
            train_loader = torch.utils.data.DataLoader(
                torch.utils.data.Subset(ds_pool, train_idx),
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
            )

            # Validation fold loader
            val_loader = torch.utils.data.DataLoader(
                torch.utils.data.Subset(ds_pool, val_idx),
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
            )

            # Train one fold
            model, best_val, _tc, _vc = train_one_run(
                train_loader, val_loader,
                base=cfg["base"], lr=cfg["lr"],
                max_epochs=60, patience=10
            )

            fold_losses.append(best_val)

        # Average validation loss across folds
        mean_loss = float(np.mean(fold_losses))
        results.append((cfg, mean_loss))
        print("Mean CV val loss:", mean_loss)

    # Pick the best configuration
    results.sort(key=lambda x: x[1])
    print("\nCV summary (best first):")
    for cfg, ml in results:
        print(cfg, "->", ml)

    return results[0][0]


# ============================================================
# 10) Plot utilities (heatmap slices + input + pred/gt)
# ============================================================

def mid_slices(vol_zyx):
    # Extract central axial, coronal and sagittal slices
    D, H, W = vol_zyx.shape
    z_mid, y_mid, x_mid = D//2, H//2, W//2
    ax = vol_zyx[z_mid]
    co = vol_zyx[:, y_mid, :]
    sa = vol_zyx[:, :, x_mid]
    return ax, co, sa

def plot_heatmap_triplet(heat_zyx, title_prefix="", vmax=1.0):
    # Plot heatmap in three orthogonal views
    ax, co, sa = mid_slices(heat_zyx)

    plt.figure(figsize=(16,4))
    for i, (img, name) in enumerate([(ax,"Axial (z mid)"), (co,"Coronal (y mid)"), (sa,"Sagittal (x mid)")]):
        plt.subplot(1,3,i+1)
        plt.imshow(img.cpu(), cmap="hot", vmin=0, vmax=vmax, aspect="auto")
        plt.title(f"{title_prefix} {name}")
        plt.colorbar()
        plt.axis("off")

    plt.tight_layout()
    plt.show()

def plot_case(volume, heat_gt, heat_pred, title=""):
    # Get central slices for input, GT heatmap and predicted heatmap
    vol_ax, _, _ = mid_slices(volume)
    gt_ax, gt_co, gt_sa = mid_slices(heat_gt)
    pr_ax, pr_co, pr_sa = mid_slices(heat_pred)

    plt.figure(figsize=(18,8))

    # Show input volume axial slice
    plt.subplot(2,4,1)
    plt.imshow(vol_ax.cpu(), cmap="gray", vmin=0, vmax=1)
    plt.title(f"{title}\nInput (axial)")
    plt.axis("off")

    # Show GT heatmap views
    for i, (img, nm) in enumerate([(gt_ax,"GT Axial"), (gt_co,"GT Coronal"), (gt_sa,"GT Sagittal")]):
        plt.subplot(2,4,2+i)
        plt.imshow(img.cpu(), cmap="hot", vmin=0, vmax=1, aspect="auto")
        plt.title(nm)
        plt.colorbar()
        plt.axis("off")

    # Show predicted heatmap views
    for i, (img, nm) in enumerate([(pr_ax,"Pred Axial"), (pr_co,"Pred Coronal"), (pr_sa,"Pred Sagittal")]):
        plt.subplot(2,4,6+i)
        plt.imshow(img.cpu(), cmap="hot", vmin=0, vmax=1, aspect="auto")
        plt.title(nm)
        plt.colorbar()
        plt.axis("off")

    plt.tight_layout()
    plt.show()

def plot_loss_curves(train_curve, val_curve):
    # Plot training and validation loss over epochs
    plt.figure(figsize=(7,5))
    plt.plot(train_curve, label="Train loss")
    plt.plot(val_curve, label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Train vs Val Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# ============================================================
# 11) Metric helpers
# ============================================================

def evaluate_heatmap_metrics(model, loader):
    """
    Compute explicit average L1 and MSE over all batches in a loader.
    """
    model.eval()
    total_l1 = 0.0
    total_mse = 0.0
    nb = 0

    with torch.no_grad():
        for X, heat_gt, _n in loader:
            X = X.to(device)
            heat_gt = heat_gt.to(device)

            # Predict heatmap
            logits = model(X)
            pred = torch.sigmoid(logits)

            # Add batch losses
            total_l1 += float(F.l1_loss(pred, heat_gt).item())
            total_mse += float(F.mse_loss(pred, heat_gt).item())
            nb += 1

    # Return average values
    return total_l1 / nb, total_mse / nb


# ============================================================
# 12) Main experiment
# ============================================================

if __name__ == "__main__":

    # -----------------------------
    # Volume size
    # -----------------------------
    D, H, W = 32, 64, 64

    # Rotation range
    angle_range_deg = 70.0

    # GT heatmap parameters
    thickness = 1.5
    sigma = 1.0
    truncate = 3.0

    align_corners = False

    # -----------------------------
    # Build disjoint rotation sets
    # -----------------------------
    # Pool is used for training/validation/CV
    angles_pool = make_angles_deg(n=240, angle_range_degrees=angle_range_deg, seed_offset=100)

    # Test set uses a separate seed so it contains unseen rotations
    angles_test = make_angles_deg(n=60,  angle_range_degrees=angle_range_deg, seed_offset=999)

    # Dataset for training/validation
    ds_pool = RotatedHeatmapDataset(
        angles_pool, D,H,W,
        thickness=thickness, sigma=sigma, truncate=truncate,
        align_corners=align_corners, device=device
    )

    # Separate test dataset
    ds_test = RotatedHeatmapDataset(
        angles_test, D,H,W,
        thickness=thickness, sigma=sigma, truncate=truncate,
        align_corners=align_corners, device=device
    )

    # -----------------------------
    # (A) Confirm GT heatmaps are correct
    # -----------------------------
    print("\n=== GT sanity check: plot + normal agreement ===")

    # Check a few GT heatmaps before training
    for i in [0, 1, 2]:
        X, heat_gt, n_true = ds_pool[i]
        heat_gt_zyx = heat_gt[0]
        print(f"\nSample {i} GT: true normal (z,y,x) = {n_true.detach().cpu().numpy()}")

        # Estimate normal back from GT heatmap
        n_est, p_est = estimate_plane_from_heatmap(heat_gt_zyx.to(device), topk=20000)
        ang = angle_between(n_est, n_true.to(device))

        print(f"Estimated normal (z,y,x) = {n_est.detach().cpu().numpy()}")
        print(f"Angle error (deg, sign-invariant) = {ang:.3f}")

        # Visualise GT heatmap views
        plot_heatmap_triplet(heat_gt_zyx, title_prefix=f"GT heatmap sample {i} | angle err {ang:.2f}°", vmax=1.0)

    # -----------------------------
    # (B) Cross-validation to pick hyperparams
    # -----------------------------
    best_cfg = run_cv(
        ds_pool,
        k=3,
        grid=[
            {"base": 24, "lr": 2e-4},
            {"base": 32, "lr": 2e-4},
            {"base": 32, "lr": 1e-4},
        ],
        batch_size=2
    )
    print("\nChosen best hyperparams:", best_cfg)

    # -----------------------------
    # (C) Final train/val split
    # -----------------------------
    # Use 80% for training and 20% for validation
    val_n = int(0.2 * len(ds_pool))
    train_n = len(ds_pool) - val_n

    train_ds, val_ds = torch.utils.data.random_split(
        ds_pool,
        [train_n, val_n],
        generator=torch.Generator().manual_seed(seed)
    )

    # Data loaders
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)
    val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=0)
    test_loader  = torch.utils.data.DataLoader(ds_test,  batch_size=2, shuffle=False, num_workers=0)

    # -----------------------------
    # (D) Train final model
    # -----------------------------
    print("\n=== Training final model ===")

    final_model, best_val, train_curve, val_curve = train_one_run(
        train_loader, val_loader,
        base=best_cfg["base"],
        lr=best_cfg["lr"],
        max_epochs=120,
        patience=18,
        w_l1=0.7, w_mse=0.3, w_grad=0.1
    )

    print("\nBest val loss:", best_val)

    # Plot train/validation loss curves
    plot_loss_curves(train_curve, val_curve)

    # -----------------------------
    # (E) Explicit TRAIN / VAL / TEST metrics
    # -----------------------------
    # Calculate final L1 and MSE on each split
    train_l1, train_mse = evaluate_heatmap_metrics(final_model, train_loader)
    val_l1, val_mse = evaluate_heatmap_metrics(final_model, val_loader)
    test_l1, test_mse = evaluate_heatmap_metrics(final_model, test_loader)

    print("\n=== TRAIN METRICS (heatmap regression) ===")
    print("Train Heat L1:", train_l1)
    print("Train Heat MSE:", train_mse)

    print("\n=== VAL METRICS (heatmap regression) ===")
    print("Val Heat L1:", val_l1)
    print("Val Heat MSE:", val_mse)

    print("\n=== TEST METRICS (heatmap regression) ===")
    print("Test Heat L1:", test_l1)
    print("Test Heat MSE:", test_mse)

    # -----------------------------
    # (F) Visualise 20 TRAINING examples + 20 TEST examples
    #     Show angle error for both training and test cases
    # -----------------------------
    print("\n=== Qualitative results: 20 training + 20 test ===")

    def infer_one(vol_zyx_1c):
        # Predict one heatmap from one input volume
        with torch.no_grad():
            logits = final_model(vol_zyx_1c[None].to(device))  # (1,1,D,H,W)
            pred = torch.sigmoid(logits)[0,0].cpu()
        return pred

    # 20 training cases
    train_indices = list(range(min(20, len(train_ds))))
    for i in train_indices:
        # Load one training example
        X, heat_gt, n_true = train_ds[i]
        vol = X[0].cpu()
        gt = heat_gt[0].cpu()

        # Predict heatmap
        pr = infer_one(X)

        # Estimate plane normal from predicted heatmap
        n_est, _p = estimate_plane_from_heatmap(pr.to(device), topk=20000)
        ang = angle_between(n_est, n_true.to(device))

        # Plot GT and prediction
        plot_case(vol, gt, pr, title=f"TRAIN case {i} | normal angle err ~ {ang:.2f}°")

    # 20 test cases
    test_indices = list(range(min(20, len(ds_test))))
    for i in test_indices:
        # Load one unseen test example
        X, heat_gt, n_true = ds_test[i]
        vol = X[0].cpu()
        gt = heat_gt[0].cpu()

        # Predict heatmap
        pr = infer_one(X)

        # Estimate plane normal from predicted heatmap
        n_est, _p = estimate_plane_from_heatmap(pr.to(device), topk=20000)
        ang = angle_between(n_est, n_true.to(device))

        # Plot GT and prediction
        plot_case(vol, gt, pr, title=f"TEST case {i} | normal angle err ~ {ang:.2f}°")

In [ ]:
# task 4

# ============================================================
# TASK 4 (FINAL SYNTHETIC PIPELINE MATCHED TO CLINICAL PIPELINE)
# Assumes Task 1, Task 2 and Task 3 have ALREADY been run.
#
# Uses:
#   - final_model
#   - train_ds, ds_test
#   - angle_between
#   - estimate_plane_from_heatmap
#
# Important:
#   - Plane point is FIXED to the centre of the 3D volume
#   - Only the NORMAL is estimated from the heatmap
#   - GT and predicted 2D slices are extracted from:
#         same fixed centre point
#         GT normal / predicted normal
#   - Display style matches the clinical pipeline:
#         heading + angle error
#         GT heatmaps overlaid on volume (axial/coronal/sagittal)
#         predicted heatmaps overlaid on volume (axial/coronal/sagittal)
#         GT slice vs predicted slice
#         RMSE only
# ============================================================

import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1) Helper functions
# ------------------------------------------------------------
def rmse_2d(a, b):
    """
    Root mean squared error between two 2D slices.
    """
    # Convert both slices to float before calculating the error
    a = a.float()
    b = b.float()

    # Calculate MSE first, then take the square root
    mse = F.mse_loss(a, b)
    rmse = torch.sqrt(mse)

    return float(rmse.item())


def plane_basis_from_normal(normal: torch.Tensor, eps: float = 1e-8):
    """
    normal: (3,) in (z,y,x)
    returns two orthonormal in-plane basis vectors u,v in (z,y,x)
    """
    # Normalise the input normal vector
    n = normal.float()
    n = n / (n.norm() + eps)

    # Pick a reference vector that is not too parallel to the normal
    ref = torch.tensor([1.0, 0.0, 0.0], device=n.device)
    if torch.abs(torch.dot(ref, n)) > 0.9:
        ref = torch.tensor([0.0, 1.0, 0.0], device=n.device)

    # First in-plane direction
    u = torch.cross(ref, n, dim=0)
    u = u / (u.norm() + eps)

    # Second in-plane direction, perpendicular to both n and u
    v = torch.cross(n, u, dim=0)
    v = v / (v.norm() + eps)

    return u, v


def extract_plane_slice_fixed_center(
    volume: torch.Tensor,      # (D,H,W) in (z,y,x)
    normal: torch.Tensor,      # (3,) in (z,y,x)
    point: torch.Tensor,       # (3,) fixed centre point in (z,y,x)
    out_h: int = 96,
    out_w: int = 96,
    spacing: float = 1.0,
    align_corners: bool = True,
):
    """
    Extract a 2D slice from a 3D volume using:
      - fixed point at the centre of the volume
      - plane normal
    """
    assert volume.ndim == 3, "volume must be (D,H,W)"
    device = volume.device
    D, H, W = volume.shape

    # Create two in-plane basis vectors from the normal
    u, v = plane_basis_from_normal(normal)

    # Create 2D coordinates for the output slice
    s = (torch.arange(out_w, device=device).float() - (out_w - 1) / 2.0) * spacing
    t = (torch.arange(out_h, device=device).float() - (out_h - 1) / 2.0) * spacing
    tt, ss = torch.meshgrid(t, s, indexing="ij")

    # Map the 2D grid into 3D space
    coords_3d = (
        point[None, None, :]
        + ss[:, :, None] * u[None, None, :]
        + tt[:, :, None] * v[None, None, :]
    )

    # Split the 3D coordinates into z, y and x
    z = coords_3d[..., 0]
    y = coords_3d[..., 1]
    x = coords_3d[..., 2]

    # Convert voxel coordinates into the normalised range used by grid_sample
    x_norm = 2.0 * (x / (W - 1)) - 1.0
    y_norm = 2.0 * (y / (H - 1)) - 1.0
    z_norm = 2.0 * (z / (D - 1)) - 1.0

    # grid_sample expects coordinates in (x,y,z) order
    grid = torch.stack([x_norm, y_norm, z_norm], dim=-1)   # (out_h,out_w,3)
    grid = grid[None, None, ...]                           # (1,1,out_h,out_w,3)

    # Add batch and channel dimensions to the volume
    vol_5d = volume[None, None].float()                    # (1,1,D,H,W)

    # Sample the volume at the plane coordinates
    sampled = F.grid_sample(
        vol_5d,
        grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=align_corners,
    )

    # Return the extracted 2D slice
    return sampled[0, 0, 0]                                # (out_h,out_w)


def infer_heatmap_one(vol_1c: torch.Tensor):
    """
    vol_1c: (1,D,H,W)
    returns predicted heatmap: (D,H,W)
    """
    # Put model in evaluation mode
    final_model.eval()

    # Predict heatmap without tracking gradients
    with torch.no_grad():
        logits = final_model(vol_1c[None].to(device))      # (1,1,D,H,W)
        pred = torch.sigmoid(logits)[0, 0].detach().cpu()

    return pred


# ------------------------------------------------------------
# 2) Plotting function
# ------------------------------------------------------------
def plot_task4_overlay_and_slices(
    volume,
    gt_heat,
    pred_heat,
    gt_slice,
    pred_slice,
    title,
    angle_err_deg,
    slice_rmse,
    point_center,
):
    """
    Display:
      Row 1: GT heatmap overlaid on volume (axial/coronal/sagittal)
      Row 2: Pred heatmap overlaid on volume (axial/coronal/sagittal)
      Row 3: GT slice / Pred slice / metrics panel
    """

    # Get volume size and centre slice indices
    D, H, W = volume.shape
    z_mid, y_mid, x_mid = D // 2, H // 2, W // 2

    # Central volume slices
    vol_ax = volume[z_mid]
    vol_co = volume[:, y_mid, :]
    vol_sa = volume[:, :, x_mid]

    # Central GT heatmap slices
    gt_ax = gt_heat[z_mid]
    gt_co = gt_heat[:, y_mid, :]
    gt_sa = gt_heat[:, :, x_mid]

    # Central predicted heatmap slices
    pred_ax = pred_heat[z_mid]
    pred_co = pred_heat[:, y_mid, :]
    pred_sa = pred_heat[:, :, x_mid]

    # Keep volume display range consistent
    vol_min = float(volume.min())
    vol_max = float(volume.max())

    fig, axes = plt.subplots(3, 3, figsize=(16, 14))
    fig.suptitle(
        f"{title}\nAngle error (deg): {angle_err_deg:.4f} | Slice RMSE: {slice_rmse:.6f}",
        fontsize=14,
        y=0.98
    )

    # ---------------- Row 1: GT overlays ----------------
    axes[0, 0].imshow(vol_ax.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max)
    im1 = axes[0, 0].imshow(gt_ax.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1)
    axes[0, 0].scatter(point_center[2].item(), point_center[1].item(), c="deepskyblue", s=60, marker="x")
    axes[0, 0].set_title("GT axial")
    axes[0, 0].axis("off")
    fig.colorbar(im1, ax=axes[0, 0], fraction=0.046, pad=0.04)

    axes[0, 1].imshow(vol_co.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max, aspect="auto")
    im2 = axes[0, 1].imshow(gt_co.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1, aspect="auto")
    axes[0, 1].scatter(point_center[2].item(), point_center[0].item(), c="deepskyblue", s=60, marker="x")
    axes[0, 1].set_title("GT coronal")
    axes[0, 1].axis("off")
    fig.colorbar(im2, ax=axes[0, 1], fraction=0.046, pad=0.04)

    axes[0, 2].imshow(vol_sa.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max, aspect="auto")
    im3 = axes[0, 2].imshow(gt_sa.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1, aspect="auto")
    axes[0, 2].scatter(point_center[1].item(), point_center[0].item(), c="deepskyblue", s=60, marker="x")
    axes[0, 2].set_title("GT sagittal")
    axes[0, 2].axis("off")
    fig.colorbar(im3, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # ---------------- Row 2: Pred overlays ----------------
    axes[1, 0].imshow(vol_ax.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max)
    im4 = axes[1, 0].imshow(pred_ax.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1)
    axes[1, 0].scatter(point_center[2].item(), point_center[1].item(), c="deepskyblue", s=60, marker="x")
    axes[1, 0].set_title("Pred axial")
    axes[1, 0].axis("off")
    fig.colorbar(im4, ax=axes[1, 0], fraction=0.046, pad=0.04)

    axes[1, 1].imshow(vol_co.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max, aspect="auto")
    im5 = axes[1, 1].imshow(pred_co.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1, aspect="auto")
    axes[1, 1].scatter(point_center[2].item(), point_center[0].item(), c="deepskyblue", s=60, marker="x")
    axes[1, 1].set_title("Pred coronal")
    axes[1, 1].axis("off")
    fig.colorbar(im5, ax=axes[1, 1], fraction=0.046, pad=0.04)

    axes[1, 2].imshow(vol_sa.cpu(), cmap="gray", vmin=vol_min, vmax=vol_max, aspect="auto")
    im6 = axes[1, 2].imshow(pred_sa.cpu(), cmap="hot", alpha=0.55, vmin=0, vmax=1, aspect="auto")
    axes[1, 2].scatter(point_center[1].item(), point_center[0].item(), c="deepskyblue", s=60, marker="x")
    axes[1, 2].set_title("Pred sagittal")
    axes[1, 2].axis("off")
    fig.colorbar(im6, ax=axes[1, 2], fraction=0.046, pad=0.04)

    # ---------------- Row 3: Slices + metric ----------------
    # Use the same display range for GT and predicted slices
    slice_min = min(float(gt_slice.min()), float(pred_slice.min()))
    slice_max = max(float(gt_slice.max()), float(pred_slice.max()))

    axes[2, 0].imshow(gt_slice.cpu(), cmap="gray", vmin=slice_min, vmax=slice_max)
    axes[2, 0].set_title("GT reformatted slice")
    axes[2, 0].axis("off")

    axes[2, 1].imshow(pred_slice.cpu(), cmap="gray", vmin=slice_min, vmax=slice_max)
    axes[2, 1].set_title("Pred reformatted slice")
    axes[2, 1].axis("off")

    # Show RMSE in the final panel
    axes[2, 2].axis("off")
    axes[2, 2].text(
        0.02, 0.85,
        f"Slice RMSE: {slice_rmse:.6f}",
        fontsize=12,
        va="top",
        ha="left"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


# ------------------------------------------------------------
# 3) Main Task 4 runner
# ------------------------------------------------------------
def run_task4_synthetic_reformatting(
    train_cases_to_show=5,
    test_cases_to_show=5,
    out_h=96,
    out_w=96,
    spacing=1.0,
    topk=20000,
):
    """
    Uses Task 3 outputs:
      - train_ds
      - ds_test
      - final_model

    For each case:
      1. predict heatmap from volume
      2. estimate GT normal from GT heatmap
      3. estimate predicted normal from predicted heatmap
      4. fix point at volume centre
      5. extract GT and predicted 2D slices
      6. display overlays + slices + metrics

    Metrics reported:
      - angle error (heatmap / normal comparison)
      - slice RMSE
    """

    # Check that the previous task outputs exist
    assert "final_model" in globals(), "final_model not found. Run Task 3 first."
    assert "train_ds" in globals(), "train_ds not found. Run Task 3 first."
    assert "ds_test" in globals(), "ds_test not found. Run Task 3 first."

    print("\n=== TASK 4 SYNTHETIC: HEATMAP -> FIXED-CENTRE REFORMATTED 2D SLICE ===")

    # ---------------- TRAIN CASES ----------------
    print("\n--- TRAINING CASES ---")
    for i in range(min(train_cases_to_show, len(train_ds))):

        # Load one training example
        X, heat_gt_1c, n_true = train_ds[i]

        # Get input volume, GT heatmap and predicted heatmap
        vol = X[0].cpu()                  # (D,H,W)
        gt_heat = heat_gt_1c[0].cpu()     # (D,H,W)
        pred_heat = infer_heatmap_one(X)  # (D,H,W)

        # Fixed centre point used for slice extraction
        D, H, W = vol.shape
        point_center = torch.tensor(
            [(D - 1) / 2.0, (H - 1) / 2.0, (W - 1) / 2.0],
            dtype=torch.float32
        )

        # Estimate normals from GT and predicted heatmaps
        gt_normal_est, _ = estimate_plane_from_heatmap(gt_heat.to(device), topk=topk)
        pred_normal_est, _ = estimate_plane_from_heatmap(pred_heat.to(device), topk=topk)

        # Move normals back to CPU
        gt_normal_est = gt_normal_est.cpu()
        pred_normal_est = pred_normal_est.cpu()

        # Calculate angle error between predicted normal and true normal
        ang_err = angle_between(pred_normal_est.to(device), n_true.to(device))

        # Extract GT slice using the GT normal
        gt_slice = extract_plane_slice_fixed_center(
            volume=vol,
            normal=gt_normal_est,
            point=point_center,
            out_h=out_h,
            out_w=out_w,
            spacing=spacing,
            align_corners=True,
        )

        # Extract predicted slice using the predicted normal
        pred_slice = extract_plane_slice_fixed_center(
            volume=vol,
            normal=pred_normal_est,
            point=point_center,
            out_h=out_h,
            out_w=out_w,
            spacing=spacing,
            align_corners=True,
        )

        # Compare predicted slice against GT slice
        slice_rmse = rmse_2d(pred_slice, gt_slice)

        # Print numerical results
        print("\n" + "=" * 90)
        print(f"TRAIN CASE {i+1}")
        print("True normal (z,y,x):      ", n_true.detach().cpu().numpy())
        print("GT-est normal (z,y,x):    ", gt_normal_est.numpy())
        print("Pred-est normal (z,y,x):  ", pred_normal_est.numpy())
        print("Angle error (deg):        ", f"{ang_err:.4f}")
        print("Slice RMSE:               ", f"{slice_rmse:.6f}")

        # Plot heatmaps and reformatted slices
        plot_task4_overlay_and_slices(
            volume=vol,
            gt_heat=gt_heat,
            pred_heat=pred_heat,
            gt_slice=gt_slice,
            pred_slice=pred_slice,
            title=f"TRAIN CASE {i+1}",
            angle_err_deg=ang_err,
            slice_rmse=slice_rmse,
            point_center=point_center,
        )

    # ---------------- TEST CASES ----------------
    print("\n--- TEST CASES ---")
    for i in range(min(test_cases_to_show, len(ds_test))):

        # Load one unseen test example
        X, heat_gt_1c, n_true = ds_test[i]

        # Get input volume, GT heatmap and predicted heatmap
        vol = X[0].cpu()                  # (D,H,W)
        gt_heat = heat_gt_1c[0].cpu()     # (D,H,W)
        pred_heat = infer_heatmap_one(X)  # (D,H,W)

        # Fixed centre point used for slice extraction
        D, H, W = vol.shape
        point_center = torch.tensor(
            [(D - 1) / 2.0, (H - 1) / 2.0, (W - 1) / 2.0],
            dtype=torch.float32
        )

        # Estimate normals from GT and predicted heatmaps
        gt_normal_est, _ = estimate_plane_from_heatmap(gt_heat.to(device), topk=topk)
        pred_normal_est, _ = estimate_plane_from_heatmap(pred_heat.to(device), topk=topk)

        # Move normals back to CPU
        gt_normal_est = gt_normal_est.cpu()
        pred_normal_est = pred_normal_est.cpu()

        # Calculate angle error between predicted normal and true normal
        ang_err = angle_between(pred_normal_est.to(device), n_true.to(device))

        # Extract GT slice using the GT normal
        gt_slice = extract_plane_slice_fixed_center(
            volume=vol,
            normal=gt_normal_est,
            point=point_center,
            out_h=out_h,
            out_w=out_w,
            spacing=spacing,
            align_corners=True,
        )

        # Extract predicted slice using the predicted normal
        pred_slice = extract_plane_slice_fixed_center(
            volume=vol,
            normal=pred_normal_est,
            point=point_center,
            out_h=out_h,
            out_w=out_w,
            spacing=spacing,
            align_corners=True,
        )

        # Compare predicted slice against GT slice
        slice_rmse = rmse_2d(pred_slice, gt_slice)

        # Print numerical results
        print("\n" + "=" * 90)
        print(f"TEST CASE {i+1}")
        print("True normal (z,y,x):      ", n_true.detach().cpu().numpy())
        print("GT-est normal (z,y,x):    ", gt_normal_est.numpy())
        print("Pred-est normal (z,y,x):  ", pred_normal_est.numpy())
        print("Angle error (deg):        ", f"{ang_err:.4f}")
        print("Slice RMSE:               ", f"{slice_rmse:.6f}")

        # Plot heatmaps and reformatted slices
        plot_task4_overlay_and_slices(
            volume=vol,
            gt_heat=gt_heat,
            pred_heat=pred_heat,
            gt_slice=gt_slice,
            pred_slice=pred_slice,
            title=f"TEST CASE {i+1}",
            angle_err_deg=ang_err,
            slice_rmse=slice_rmse,
            point_center=point_center,
        )


# ------------------------------------------------------------
# 4) Run Task 4
# ------------------------------------------------------------
run_task4_synthetic_reformatting(
    train_cases_to_show=5,   # number of training examples to display
    test_cases_to_show=5,    # number of test examples to display
    out_h=96,
    out_w=96,
    spacing=1.0,
    topk=20000,
)